# 第45章：离线生产工程构造示例

默认只新跑一个正常构造场景，写SQLite状态与核心产物，核现金/NAV/PnL后新画一图；不演示完整崩溃、停止、恢复和幂等回放。共用已接受核心具备这些独立工程检查，但不能据此声称本Notebook执行了全部恢复套件。

UTC事件、可得、入库、as_of均须在信号前合格，成交后于信号。类型、单位、新鲜度和有限值在成交前校验。固定配置允许正实数股数，不代表券商lot size，金额容差1e−6 USD；非正动量目标为0。无训练、市场adapter或部署。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch45_production_quant_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import csv, json
import matplotlib.pyplot as plt
from quant_course.production_pipeline import default_config, constructed_inputs, run_pipeline


## 1. 检查默认配置与构造输入

流水线输入显式声明货币、单位、版本以及观测与未来撮合的严格切分。

In [ ]:
cfg = default_config()
inputs = constructed_inputs()

print("Initial Cash:", cfg["initial_cash"], cfg["currency"])
print("Half Spread (bps):", cfg["execution"]["half_spread_bps"])
print("Commission Rate:", cfg["execution"]["commission_rate"])
print("Securities metadata:", inputs["metadata"]["securities"])
print("Observations count:", len(inputs["observations"]))
print("Signal Time:", inputs["metadata"]["signal_time"])

## 2. 执行一个正常场景

run_pipeline仅调用一次；key=nb_demo_run，state.db和artifacts均位于新RUN_DIR。输入闸门、动量/事前风险、逐笔持久化、对账与暂存发布由共用核心完成。本格没有注入故障、STOPPED或重放。绘图为核心发布后的附属工件，不在同一事务中。

In [ ]:
run_dir = RUN_DIR
state_db, output_dir = run_dir / "state.db", run_dir / "artifacts"
summary = run_pipeline(config=cfg, inputs=inputs, run_key="nb_demo_run",
                       state_db=state_db, output_dir=output_dir)
print(json.dumps(summary, indent=2))
print("Preserved run:", run_dir.relative_to(ROOT))


## 3. 结果核验与独立平衡表对账

验证各项关键指标与容差残差：
- 最终现金：$7995.998$ USD
- 最终净值：$9995.998$ USD
- 交易成本：点差 $2.000$ + 佣金 $2.002 = 4.002$ USD
- 毛收益 $0.000$ USD，净收益 $-4.002$ USD
- 平衡表对账状态：PASS（实际残差在声明浮点容差范围内）

In [ ]:
assert abs(summary["final_cash"] - 7995.998) <= 1e-6
assert abs(summary["terminal_nav"] - 9995.998) <= 1e-6
assert abs(summary["net_pnl"] + 4.002) <= 1e-6
assert summary["reconciliation"]["status"] == "PASS"
assert summary["reconciliation"]["cash_reconciled"] is True
assert summary["reconciliation"]["positions_reconciled"] is True
assert summary["reconciliation"]["nav_pnl_reconciled"] is True
assert summary["gate_outcomes"]["balance_sheet_reconciliation"] == "PASS"
print("CONSTRUCTED baseline agrees with the hand calculation; author candidate only.")

## 4. 单期内的事件阶段图

读取ledger_events.csv按阶段画现金/NAV，不是月频多期研究。核心正常产生CSV；代码另有缺少记录时显示初末值的fallback。新图写当前RUN_DIR。

In [ ]:
ledger_csv_path = output_dir / "ledger_events.csv"
stages = []
cash_series = []
nav_series = []

if ledger_csv_path.exists():
    with open(ledger_csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            evt = row.get("event_type", "EVENT")
            ts = row.get("timestamp", "")
            short_time = ts[11:16] if len(ts) >= 16 else ts
            stages.append(f"{evt}\n({short_time})")
            cash_series.append(float(row.get("cash_after", 0.0)))
            nav_series.append(float(row.get("nav", 0.0)))

if not stages:
    stages = ["Initial", "Final"]
    cash_series = [summary["initial_cash"], summary["final_cash"]]
    nav_series = [summary["initial_cash"], summary["terminal_nav"]]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(stages, nav_series, marker="o", color="#2980b9", linewidth=2, label="NAV (USD)")
ax.plot(stages, cash_series, marker="s", color="#e67e22", linewidth=2, linestyle="--", label="Cash (USD)")
ax.set_ylabel("USD")
ax.set_title("CONSTRUCTED: Cash & NAV by recorded event", fontweight="bold")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="center right")
plt.tight_layout()
plt.savefig(RUN_DIR / "ch45_production_quant_cell09.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()